# Experiment 007 — Jacobian Residual Repair (JRR)

This notebook tests whether strong activation steering fails because of a downstream nonlinear Taylor remainder. It deliberately separates calibration diagnostics from held-out evaluation.


In [ ]:
import os, pathlib, subprocess, sys
repo = pathlib.Path('/content/steering-manifold-repair')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/Nek1tt/steering-manifold-repair.git',str(repo)], check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'], check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'], check=True)
print('cwd:', os.getcwd())


## 0. Restore the frozen steering direction if needed

This does not tune JRR. It only reconstructs the already-established sentiment baseline direction in a fresh runtime.


In [ ]:
from pathlib import Path
direction = Path('results/sentiment_direction.pt')
if not direction.exists():
    subprocess.run([sys.executable,'scripts/validate_sentiment_baseline.py','--config','configs/baseline_sentiment_gpt2.yaml'], check=True)
else:
    print('Using frozen direction:', direction)


## 1. Fast mathematical/unit checks


In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_jrr.py','tests/test_inference_followups.py','tests/test_denoiser.py'], check=True)


## 2. Real-model JVP preflight

This is a numerical safety check, not a scientific result. It verifies on a real GPT-2 prompt that the primary JVP agrees with an independent central finite difference at every candidate downstream hook. Do not continue if it fails.


In [ ]:
subprocess.run([sys.executable,'scripts/preflight_jrr.py','--config','configs/jrr_gpt2.yaml'], check=True)


## 3. Stage A — nonlinear propagation diagnostic

Uses calibration prompts only. This is the decisive cheap test before any expensive oracle generation.


In [ ]:
subprocess.run([sys.executable,'scripts/run_jrr_diagnostic.py','--config','configs/jrr_gpt2.yaml'], check=True)


In [ ]:
import json, pandas as pd
from IPython.display import display, Image
summary = json.loads(Path('results/jrr/diagnostic_summary.json').read_text())
print(json.dumps(summary, indent=2))
display(pd.read_csv('results/jrr/target_layer_summary.csv'))
display(Image(filename='results/jrr/remainder_scaling.png'))
display(Image(filename='results/jrr/orthogonal_remainder_fraction.png'))
display(Image(filename='results/jrr/orthogonal_remainder_vs_fluency.png'))


## 4. Stage B — exact causal oracle on calibration prompts

Run this only when `oracle_recommended` above is true. The script itself enforces the gate.


In [ ]:
assert summary['oracle_recommended'], 'Diagnostic did not support spending compute on oracle JRR. Stop here and report the negative result.'
subprocess.run([sys.executable,'scripts/run_jrr_oracle.py','--config','configs/jrr_gpt2.yaml','--phase','calibration'], check=True)


In [ ]:
cal = json.loads(Path('results/jrr/oracle_calibration_summary.json').read_text())
print(json.dumps(cal, indent=2))
display(pd.read_csv('results/jrr/oracle_calibration_frontier.csv'))
display(Image(filename='results/jrr/oracle_calibration_pareto.png'))


## 5. Frozen held-out evaluation

Only run after oracle calibration passes. No tuning is allowed after this cell.


In [ ]:
assert cal['go_to_heldout'], 'Calibration oracle did not beat the frozen gate. Do not touch held-out data.'
subprocess.run([sys.executable,'scripts/run_jrr_oracle.py','--config','configs/jrr_gpt2.yaml','--phase','evaluation'], check=True)


In [ ]:
display(pd.read_csv('results/jrr/oracle_evaluation_frontier.csv'))
display(Image(filename='results/jrr/oracle_evaluation_pareto.png'))
print(Path('results/jrr/oracle_evaluation_summary.json').read_text())


## What to send back for analysis

Zip `results/jrr/` after whichever stage you reached. A negative Stage A or Stage B result is scientifically useful; do not bypass the gates just to obtain a held-out graph.


In [ ]:
import shutil
archive = shutil.make_archive('/content/jrr_results', 'zip', 'results/jrr')
print('Created:', archive)
